# ECON417 — Project 2: Banking Analytics
## Predicting Bank Profitability (Return on Assets)

**Domain:** Finance — Banking
**Dataset:** FDIC Bank Financial Data (U.S. Commercial Banks)
**Research Question:** Which bank financial characteristics best predict Return on Assets (ROA)?

---

### Learning Objectives
1. Download and engineer features from a real federal regulatory dataset
2. Apply demeaning and understand its meaning in a banking context
3. Compare OLS, Ridge, and Lasso for financial ratio regression
4. Use Random Forest to capture nonlinear relationships in bank performance

### Background

The **Federal Deposit Insurance Corporation (FDIC)** is the primary federal regulator
and deposit insurer for U.S. commercial banks. The FDIC publishes quarterly financial
data for every insured institution — over 4,500 banks — through its open BankFind Suite API.

**Return on Assets (ROA)** is the most widely used metric for bank profitability:

$$ROA = \frac{\text{Net Income}}{\text{Total Assets}} \times 100\%$$

| ROA Level | Industry Interpretation |
|-----------|------------------------|
| > 1.5% | Excellent |
| 1.0% – 1.5% | Good |
| 0% – 1.0% | Below average |
| < 0% | Unprofitable |

Understanding what drives ROA is central to:
- **Regulatory exams** (CAMELS rating system)
- **Bank equity research** at investment banks and asset managers
- **Credit analysis** at ratings agencies (Moody's, S&P, Fitch)

**Data source:** FDIC BankFind Suite API — https://banks.data.fdic.gov/api/
(U.S. government open data, no authentication required)


## Section 0 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style="whitegrid")
print("All libraries loaded successfully!")

## Section 1 — Data Acquisition

We download year-end 2023 financial data for all U.S. commercial banks
directly from the FDIC BankFind Suite API.

**API endpoint:** `https://banks.data.fdic.gov/api/financials`
**Documentation:** https://banks.data.fdic.gov/docs/

**Raw fields we request:**

| Field | Description |
|-------|-------------|
| `asset` | Total assets (thousands of USD) |
| `netinc` | Net income |
| `eq` | Total equity capital |
| `intinc` | Total interest income |
| `nonii` | Total noninterest income |
| `lnlsnet` | Net loans and leases |

In [ ]:
import requests
from io import StringIO

api_url = (
    "https://banks.data.fdic.gov/api/financials"
    "?fields=repdte,asset,netinc,eq,intinc,nonii,lnlsnet"
    "&filters=repdte%3A20231231"
    "&limit=5000"
    "&output=csv"
)

try:
    print("Downloading FDIC bank financial data (year-end 2023)...")
    resp = requests.get(api_url, timeout=30)
    resp.raise_for_status()
    df_raw = pd.read_csv(StringIO(resp.text))
    print(f"Downloaded data for {len(df_raw):,} U.S. banks.")
    print(f"Columns: {list(df_raw.columns)}")
except Exception as e:
    print(f"Download failed ({e}). Using simulated FDIC-compatible data...")
    np.random.seed(42)
    n = 3000
    asset_base     = np.random.lognormal(mean=10, sigma=2.2, size=n)
    equity_ratio   = np.random.beta(8, 52, size=n)
    loan_ratio     = np.clip(np.random.beta(6, 4, size=n) * 0.6 + 0.2, 0.05, 0.90)
    int_ratio      = loan_ratio * np.random.uniform(0.03, 0.06, n)
    nonint_ratio   = np.random.uniform(0.004, 0.025, size=n)
    roa_true       = (2.4 * equity_ratio + 1.6 * loan_ratio
                      - 0.9 * nonint_ratio
                      + 0.3 * np.log1p(asset_base) / 20
                      + np.random.normal(0, 0.28, n))
    df_raw = pd.DataFrame({
        'asset':   asset_base * 1000,
        'netinc':  asset_base * roa_true / 100,
        'eq':      asset_base * equity_ratio,
        'intinc':  asset_base * int_ratio,
        'nonii':   asset_base * nonint_ratio,
        'lnlsnet': asset_base * loan_ratio,
    })

df_raw.head()

## Section 2 — Feature Engineering

Raw FDIC fields are dollar amounts — they vary with bank size, making direct
comparison meaningless. Bank analysts always convert to **financial ratios**
that scale out size effects. This mirrors the CAMELS regulatory framework.

| Engineered Feature | Formula | Economic Meaning |
|--------------------|---------|-----------------|
| `equity_ratio` | Equity / Assets | Capital adequacy (safety buffer) |
| `loan_ratio` | Net Loans / Assets | Core business intensity |
| `interest_ratio` | Interest Income / Assets | Yield on earning assets |
| `nonint_ratio` | Noninterest Income / Assets | Fee-based revenue diversification |
| `log_asset` | log(1 + Assets) | Bank size (log-transformed for skew) |
| **`roa`** | Net Income / Assets × 100 | **Target: profitability (%)** |

In [ ]:
df = df_raw.copy()

# Target variable
df['roa']           = (df['netinc']   / df['asset']) * 100

# Predictor features (all ratios → size-invariant)
df['equity_ratio']  = df['eq']      / df['asset']
df['loan_ratio']    = df['lnlsnet'] / df['asset']
df['interest_ratio']= df['intinc']  / df['asset']
df['nonint_ratio']  = df['nonii']   / df['asset']
df['log_asset']     = np.log1p(df['asset'])

# Clean: remove inf, NaN, and extreme ROA outliers (data errors)
df = df.replace([np.inf, -np.inf], np.nan).dropna()
df = df[(df['roa'] > -5) & (df['roa'] < 8)]
df = df[df['asset'] > 0]

feature_cols = ['equity_ratio', 'loan_ratio', 'interest_ratio', 'nonint_ratio', 'log_asset']
X = df[feature_cols].copy()
y = df['roa'].copy()

print(f"Final sample: {len(df):,} banks")
print(f"\nTarget (ROA) summary:")
print(f"  Mean   = {y.mean():.4f}%")
print(f"  Median = {y.median():.4f}%")
print(f"  Std    = {y.std():.4f}%")
X.describe().round(4)

## Section 3 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(y, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(y.mean(),   color='red',    linestyle='--', lw=2,
                label=f'Mean   = {y.mean():.3f}%')
axes[0].axvline(y.median(), color='orange', linestyle='--', lw=2,
                label=f'Median = {y.median():.3f}%')
axes[0].set_xlabel('Return on Assets (%)')
axes[0].set_ylabel('Number of Banks')
axes[0].set_title('Distribution of Bank ROA — U.S. Insured Institutions (2023)')
axes[0].legend()

axes[1].boxplot(y, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='red', lw=2),
                flierprops=dict(marker='o', markersize=3, alpha=0.3))
axes[1].set_ylabel('ROA (%)')
axes[1].set_title('ROA Distribution — Boxplot')

plt.tight_layout()
plt.show()
print(f"Skewness: {y.skew():.3f}  |  Kurtosis: {y.kurt():.3f}")

In [ ]:
# Correlation heatmap
data_full = X.copy()
data_full['roa'] = y
corr = data_full.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Banking Features — Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

print("Correlation with ROA (sorted):")
corrs = corr['roa'].drop('roa').abs().sort_values(ascending=False)
for feat, val in corrs.items():
    print(f"  {feat:20s}: {val:.4f}")

In [ ]:
# Scatter plots: each feature vs ROA
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for i, feat in enumerate(feature_cols):
    axes[i].scatter(X[feat], y, alpha=0.25, s=8, color='steelblue', edgecolors='none')
    z = np.polyfit(X[feat], y, 1)
    x_line = np.linspace(X[feat].min(), X[feat].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), 'r-', lw=2.5)
    axes[i].set_xlabel(feat, fontsize=9)
    axes[i].set_ylabel('ROA (%)' if i == 0 else '')
    axes[i].set_title(feat, fontsize=10)
plt.suptitle('Feature vs ROA — U.S. Commercial Banks', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Section 4 — Demeaning

In a banking context, demeaning reframes each ratio as a **deviation from the
industry average**. After demeaning:

- The OLS intercept equals the **industry average ROA**
- Each coefficient represents the effect on ROA of being *one unit above the
  industry average* on that ratio
- This framing is standard in bank performance benchmarking reports
  (e.g., FDIC Quarterly Banking Profile)

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Compute X_mean = X.mean() and X_demeaned = X - X_mean. Print a table comparing original means to demeaned means to verify.
raise NotImplementedError()

## Section 5 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_demeaned, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]:,} banks  |  Test set: {X_test.shape[0]:,} banks")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

results_all = []
print("Standardized features ready for Ridge and Lasso.")

## Section 6 — Evaluation Function

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Define evaluate_model(y_true, y_pred, model_name) that computes and returns R2, RMSE, and MAE using r2_score(), mean_squared_error(), mean_absolute_error()
raise NotImplementedError()

## Section 7 — OLS Regression

OLS assumes ROA is a linear function of the financial ratios:

$$ROA_i = \beta_0 + \beta_1 \cdot EqRatio_i + \beta_2 \cdot LoanRatio_i + \ldots + \varepsilon_i$$

Because we demeaned, $\hat{\beta}_0$ directly represents the
**average ROA across all U.S. banks**.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Fit LinearRegression() on (X_train, y_train), predict on X_test, call evaluate_model(), store result in lr_res.
raise NotImplementedError()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_test, y_pred_lr, alpha=0.35, s=18, color='steelblue', edgecolors='none')
lims = [min(y_test.min(), y_pred_lr.min()), max(y_test.max(), y_pred_lr.max())]
axes[0].plot(lims, lims, 'r--', lw=2)
axes[0].set_xlabel('Actual ROA (%)')
axes[0].set_ylabel('Predicted ROA (%)')
axes[0].set_title('OLS — Predicted vs Actual')

resid_lr = y_test.values - y_pred_lr
axes[1].scatter(y_pred_lr, resid_lr, alpha=0.35, s=18, color='teal', edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted ROA (%)')
axes[1].set_ylabel('Residuals')
axes[1].set_title('OLS — Residual Plot')

c_colors = ['steelblue' if c > 0 else 'coral' for c in lr.coef_]
axes[2].barh(feature_cols, lr.coef_, color=c_colors, edgecolor='white', alpha=0.87)
axes[2].axvline(0, color='black', lw=1)
axes[2].set_xlabel('Coefficient Value')
axes[2].set_title('OLS — Coefficients')

plt.tight_layout()
plt.show()

## Section 8 — Ridge Regression (L2 Regularization)

Bank financial ratios are often correlated — for example, loan ratio and interest income
ratio both reflect lending activity. Ridge regression handles this multicollinearity by
shrinking all coefficients simultaneously.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Use RidgeCV(alphas=np.logspace(-3,3,100), cv=5) to find the best alpha on X_train_scaled. Fit Ridge(alpha=best_alpha_ridge) and predict on X_test_scaled. Call evaluate_model() and store in ridge_res.
raise NotImplementedError()

In [ ]:
alpha_range = np.logspace(-3, 3, 60)
paths_r = np.array([
    Ridge(alpha=a).fit(X_train_scaled, y_train).coef_ for a in alpha_range
])

plt.figure(figsize=(11, 5))
for i, feat in enumerate(feature_cols):
    plt.semilogx(alpha_range, paths_r[:, i], lw=2, label=feat)
plt.axvline(best_alpha_ridge, color='black', ls='--', lw=2.5,
            label=f'Best α = {best_alpha_ridge:.3f}')
plt.xlabel('α  (log scale)')
plt.ylabel('Coefficient Value')
plt.title('Ridge — Coefficient Path (Bank Financial Features)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## Section 9 — Lasso Regression (L1 Regularization)

In practice, a banking analyst might collect dozens of financial ratios. Lasso answers:
**which ratios actually matter for predicting ROA?** A coefficient driven to zero by
Lasso suggests that ratio is redundant given the others in the model.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Use LassoCV(alphas=np.logspace(-3,3,100), cv=5, max_iter=10000) to find the best alpha. Fit Lasso(alpha=best_alpha_lasso, max_iter=10000) on X_train_scaled. Predict, call evaluate_model(), store in lasso_res. Print feature selection results.
raise NotImplementedError()

In [ ]:
# Coefficient comparison across the three linear models
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (coef, title, color) in zip(axes, [
    (lr.coef_,    'OLS Coefficients',   'steelblue'),
    (ridge.coef_, 'Ridge Coefficients', 'teal'),
    (lasso.coef_, 'Lasso Coefficients', 'coral'),
]):
    c_colors = [color if c > 0 else '#c62828' for c in coef]
    ax.barh(feature_cols, coef, color=c_colors, edgecolor='white', alpha=0.87)
    ax.axvline(0, color='black', lw=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Coefficient')
plt.suptitle('Coefficient Comparison — Banking Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 10 — Linear Models Comparison

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Build a DataFrame from [lr_res, ridge_res, lasso_res], set 'Model' as index, round to 4 decimal places, and print the comparison table.
raise NotImplementedError()

## Section 11 — Random Forest

Bank profitability involves nonlinear dynamics:
- **High equity + high loans → high ROA**, but
- **Low equity + high loans → elevated risk and potential losses**

These interaction effects are captured naturally by decision trees within
the Random Forest ensemble, but are invisible to linear models.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Create RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=10, random_state=42, n_jobs=-1), fit on (X_train, y_train), predict on X_test, call evaluate_model(), store in rf_res.
raise NotImplementedError()

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Extract rf.feature_importances_, create importance_df with columns ['Feature', 'Importance'], sort descending by Importance.
raise NotImplementedError()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

palette_rf = sns.color_palette("Blues_r", len(importance_df))
axes[0].barh(importance_df['Feature'][::-1],
             importance_df['Importance'][::-1],
             color=palette_rf, edgecolor='white', alpha=0.9)
axes[0].set_xlabel('Feature Importance')
axes[0].set_title('Random Forest — Feature Importance
(Bank ROA Prediction)')

axes[1].scatter(y_test, y_pred_rf, alpha=0.35, s=18, color='teal', edgecolors='none')
lims = [min(y_test.min(), y_pred_rf.min()), max(y_test.max(), y_pred_rf.max())]
axes[1].plot(lims, lims, 'r--', lw=2)
axes[1].set_xlabel('Actual ROA (%)')
axes[1].set_ylabel('Predicted ROA (%)')
axes[1].set_title('Random Forest — Predicted vs Actual ROA')
r2_rf = r2_score(y_test, y_pred_rf)
axes[1].text(0.05, 0.95, f'R² = {r2_rf:.4f}', transform=axes[1].transAxes,
             fontsize=12, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Section 12 — Final Comparison and Conclusion

In [ ]:
results_final = pd.DataFrame(results_all).set_index('Model').round(4)
print("All Models — Final Summary:")
print("="*58)
print(results_final.to_string())
print("="*58)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
palette = ['#1565C0', '#00695C', '#BF360C', '#2E7D32']
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = results_final[metric]
    bars = axes[i].bar(vals.index, vals, color=palette[:len(vals)],
                       edgecolor='white', alpha=0.88)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2.,
                     bar.get_height() + max(vals)*0.01,
                     f'{val:.3f}', ha='center', va='bottom',
                     fontsize=9, fontweight='bold')
    title_map = {'R2': 'R²', 'RMSE': 'RMSE', 'MAE': 'MAE'}
    axes[i].set_title(title_map[metric], fontsize=12)
    axes[i].set_ylabel(title_map[metric])
    axes[i].tick_params(axis='x', rotation=20)
plt.suptitle('Banking Project — All Models Performance Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusions and Industry Implications

### Model Performance Takeaways

**OLS (Baseline):** Provides transparent, coefficient-based insights — directly
actionable for writing bank research reports or regulatory memos.

**Ridge:** More robust to the multicollinearity common in financial ratios
(e.g., loan ratio and interest income ratio are highly correlated).

**Lasso:** Identifies the minimal set of ratios needed for accurate ROA prediction —
useful for building streamlined early-warning scorecards.

**Random Forest:** Captures nonlinear interactions (e.g., leverage × loan
concentration effects) that drive large differences in bank profitability.

### Key Findings

- **Interest income ratio** and **equity ratio** are typically the strongest predictors
- This aligns with banking fundamentals: net interest margin is the primary revenue
  driver, and capital adequacy determines risk capacity
- **Noninterest income ratio** may be zeroed out by Lasso — fee income is noisier
  and less consistently correlated with ROA

### Practical Applications in Banking

1. **FDIC / OCC examination preparation** — Identify institutions whose predicted ROA
   deviates significantly from actual ROA (potential for misclassification or fraud)
2. **Peer group benchmarking** — Demean features by peer-group means for more
   meaningful comparisons (community banks vs. regional banks)
3. **Stress testing** — Simulate how ROA changes under adverse macroeconomic scenarios
   by shifting feature values

### Discussion Questions

> 1. The SVB (Silicon Valley Bank) collapse in March 2023 was partly driven by
>    an unusually low equity-to-assets ratio and concentrated deposits.
>    Would the features in this model have provided an early warning signal?
>
> 2. Why is ROA preferred over net income as a performance target in this model?
>    What normalization principle does this reflect?
>
> 3. What are the limitations of cross-sectional regression (all banks at one point in
>    time) compared to a panel regression (tracking the same banks over many quarters)?